In [0]:
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0
!pip install "transformers[torch]"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from pyspark.sql.functions import col, to_timestamp, from_utc_timestamp, date_format, round
from transformers import pipeline
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StringType, FloatType, StructType, StructField
import pandas as pd
import scipy
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [0]:
jdbc_url = dbutils.secrets.get(scope="Capstone", key="DatabasejdbcUrl")#DatabasejdbcUrl
connection_properties = {
    "user": dbutils.secrets.get(scope="Capstone", key="DatabaseUsername"),
    "password": dbutils.secrets.get(scope="Capstone", key="DatabasePassword"),
    "driver": dbutils.secrets.get(scope="Capstone", key="DatabaseDriver")
}

old_data = spark.read.jdbc(
    url=jdbc_url,
    table="Gold.Historical_Stock_News_Sentiment_Score",
    properties=connection_properties
).orderBy("id")


new_data = spark.read.jdbc(
    url=jdbc_url,
    table="Silver.Historical_Stock_News",
    properties=connection_properties
).orderBy("id")

diff_data = new_data.join(old_data, on="id", how="left_anti")
diff_data = diff_data.withColumn("datetime", to_timestamp("datetime"))
diff_data = diff_data.withColumn("datetime", from_utc_timestamp("datetime", "America/New_York"))
diff_data = diff_data.withColumn("datetime", date_format("datetime", "yyyy-MM-dd HH:mm:ss"))
display(diff_data)

In [0]:
X = diff_data.toPandas()['headline'].to_list()
list_id = diff_data.toPandas()['id'].to_list()

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

counter = 0

preds_proba = []
tokenizer_kwargs = {"padding": True, "truncation": True, "max_length": 512}

for x in X:
    with torch.no_grad():
        input_sequence = tokenizer(x, return_tensors="pt", **tokenizer_kwargs)
        logits = model(**input_sequence).logits
        scores = {
        k: v
        for k, v in zip(
            model.config.id2label.values(),
            scipy.special.softmax(logits.numpy().squeeze()),
        )
    }
    result_sentiment_list = list(scores.values())
    result_sentiment_list.append(list_id[counter])
    preds_proba.append(result_sentiment_list)
    counter += 1

cleaned_preds_proba = [[float(p), float(n), float(neu), int(row_id)] for p, n, neu, row_id in preds_proba]

schema_sentiment = StructType([
    StructField("positive_value", FloatType(), True),
    StructField("negative_value", FloatType(), True),
    StructField("neutral_value",  FloatType(), True),
    StructField("id",         IntegerType(), True)
])

df_sentiment = spark.createDataFrame(cleaned_preds_proba, schema=schema_sentiment)
new_data_with_score = diff_data.join(df_sentiment, on="id", how="inner").orderBy("id")

In [0]:
new_data_with_score.display()

In [0]:
new_data_with_score = new_data_with_score.withColumn("datetime", to_timestamp("datetime"))
new_data_with_score = new_data_with_score.withColumn("datetime", from_utc_timestamp("datetime", "America/New_York"))
new_data_with_score = new_data_with_score.withColumn("datetime", date_format("datetime", "yyyy-MM-dd HH:mm:ss"))

new_data_with_score.count()

In [0]:
new_data_with_score.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Gold.Historical_Stock_News_Sentiment_Score") \
    .option("user", connection_properties["user"]) \
    .option("password", connection_properties["password"]) \
    .option("driver", connection_properties["driver"]) \
    .mode("append") \
    .option("batchsize", 10000) \
    .option("numPartitions", 8) \
    .save()

print("Data successfully written to Azure SQL Database.")

In [0]:
# CREATE TABLE Gold.Historical_Stock_News_Sentiment_Score (
#     category        NVARCHAR(255),
#     datetime        NVARCHAR(50),
#     headline        NVARCHAR(MAX),
#     image           NVARCHAR(2083),
#     related         NVARCHAR(255),
#     source          NVARCHAR(255),
#     summary         NVARCHAR(MAX),
#     url             NVARCHAR(2083),
#     symbol          NVARCHAR(50),
#     positive_value  FLOAT,
#     negative_value  FLOAT,
#     neutral_value   FLOAT
# );